In [0]:
from pyspark.sql import functions as F

silver_path = "/Volumes/workspace/default/filestore/day12/silver/orders"
gold_path = "/Volumes/workspace/default/filestore/day12/gold/orders"

silver_df = spark.read.format("delta").load(silver_path)

display(silver_df)

order_id,customer_id,order_date,amount
102,1,2026-09-21,1200.0
104,3,2026-09-22,2500.0
101,1,2026-09-20,500.0
103,2,2026-09-21,800.0
105,2,2026-09-23,1500.0


In [0]:
gold_df = (
    silver_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("amount").alias("total_revenue"),
        F.sum("amount").alias("revenue"),
        F.avg("amount").alias("avg_order_value"),
        F.max("order_date").alias("last_order_date")
    )
)

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

print("Gold aggregation completed")

Gold aggregation completed


In [0]:
final_gold_df = spark.read.format("delta").load(gold_path)

display(final_gold_df.orderBy("customer_id"))

customer_id,total_orders,total_revenue,revenue,avg_order_value,last_order_date
1,2,1700.0,1700.0,850.0,2026-09-21
2,2,2300.0,2300.0,1150.0,2026-09-23
3,1,2500.0,2500.0,2500.0,2026-09-22
